In [ ]:
# Install required packages
!pip install pandas openpyxl

In [ ]:
import pandas as pd
import re
from typing import Dict, List

In [ ]:
# Load from Google Drive
from google.colab import drive
drive.mount('/content/drive')

taste_df = pd.read_excel('/content/drive/MyDrive/dataset/taste_description.xlsx')
emotion_df = pd.read_excel('/content/drive/MyDrive/dataset/emotion_description.xlsx')

taste_df.head()
emotion_df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Unnamed: 0,angularity,roundness,symmetry,elements,hue,brightness,saturation,texture,complexity
0,happy,Low to Medium,High,Medium,Medium,"Yellow, Orange",High,High,Smooth/ Soft,Medium (Playful)
1,pleasure,Low,Very High,High (D8/Radial),Low to Medium,"Pink, Warm Red",High,Medium,Smooth/Creamy,Low
2,asthonished,High (Spiky/Starbursts),Low,Low or Radial,High,"Neon Green, Bright Yellow, High Contrast",High,High,Sharp/Rough,High (Unexpected elements)
3,relaxed,Low,High,High,Few (Minimalist),"Light Blue, White",High,Low (Pastel),Smooth/Silky,Low
4,satisfied,Low,Medium,High,Low,"Beige, Warm Earth",Medium-High,Medium,Matte / Natural,Low


#Define Template Formats

In [ ]:
TEMPLATE = \
    "A book cover that features {angularity} angularity, " \
    "{roundness} roundness, {symmetry} symmetry, {elements} elements, " \
    "with {hue} hues, {brightness} brightness, {saturation} saturation, " \
    "{texture} textures, and {complexity} complexity."

In [ ]:
print(type(TEMPLATE))
print(TEMPLATE)

<class 'str'>
A book cover that features {angularity} angularity, {roundness} roundness, {symmetry} symmetry, {elements} elements, with {hue} hues, {brightness} brightness, {saturation} saturation, {texture} textures, and {complexity} complexity.


In [ ]:
def clean_value(value):
    """
    Clean and normalize field values
    """
    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Convert to lowercase for consistency (optional)
    # value = value.lower()

    return value

class FallbackDict(dict):
    """Returns empty string for any missing key instead of raising KeyError."""
    def __missing__(self, key):
        return ""

def fill_template(row: pd.Series) -> str:
    values = {col: clean_value(row.get(col, "")) for col in row.index}

    # Only include fields that have values
    shape_parts = []
    if values.get('angularity'): shape_parts.append(f"{values['angularity']} angularity")
    if values.get('roundness'):  shape_parts.append(f"{values['roundness']} roundness")
    if values.get('symmetry'):   shape_parts.append(f"{values['symmetry']} symmetry")
    if values.get('elements'):   shape_parts.append(f"{values['elements']} elements")

    color_parts = []
    if values.get('hue'):        color_parts.append(f"{values['hue']} hues")
    if values.get('brightness'): color_parts.append(f"{values['brightness']} brightness")
    if values.get('saturation'): color_parts.append(f"{values['saturation']} saturation")

    style_parts = []
    if values.get('texture'):    style_parts.append(f"{values['texture']} textures")
    if values.get('complexity'): style_parts.append(f"{values['complexity']} complexity")

    # Build sentence only from available parts
    parts = []
    if shape_parts: parts.append(", ".join(shape_parts))
    if color_parts: parts.append(", ".join(color_parts))
    if style_parts: parts.append(", ".join(style_parts))

    if not parts:
        return ""

    return "A book cover that features " + ", ".join(parts) + "."

def generate_descriptions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()  # avoid mutating the original
    df['description'] = df.apply(fill_template, axis=1)
    return df

In [ ]:
# Generate descriptions
taste_desc = generate_descriptions(taste_df)
emotion_desc = generate_descriptions(emotion_df)

print("Taste Descriptions:")
for idx, row in taste_desc.head(5).iterrows():
    print(f"\n{idx + 1}. {row['description']}")

print("\nEmotion Descriptions:")
for idx, row in emotion_desc.iterrows():
    print(f"\n{idx + 1}. {row['description']}")

Taste Descriptions:

1. A book cover that features Low (Curved, Smooth) angularity, High (Spherical, Circular) roundness, High (Reflectional/Rotational) symmetry, Low to Medium elements, Pink / Red hues, High (Light/Pastel) brightness, High (Vivid/Neon) - Candy like saturation, Smooth / Soft textures, Low (Simple/Clean) complexity.

2. A book cover that features High (Sharp, Spiky, Triangular) angularity, Low (Asymmetrical) symmetry, Many (High Clutter) elements, Green / Yellow hues, High (Vivid/Neon) saturation, Rough / Gritty / Jagged textures.

3. A book cover that features Low to Medium angularity, Low (Flat, Linear) roundness, Low to Medium symmetry, Few (Minimalist) - if simple/clean elements, Blue / White hues, High (Light/Pastel) brightness, Low (Desaturated/Grey) saturation, Crystalline / Rough textures, Low (Simple/Clean) complexity.

4. A book cover that features Very High (Sharp, Spiky, Triangular) angularity, Low roundness, Very Low (Asymmetrical) symmetry, Many (High Clut

In [ ]:
taste_descriptions = dict(zip(taste_desc.index, taste_desc['description']))
emotion_descriptions = dict(zip(emotion_desc.index, emotion_desc['description']))

def count_tokens(text: str) -> int:
    """Simple token counter that splits by words and punctuation."""
    tokens = re.findall(r'\w+|[^\w\s]', text)
    return len(tokens)

print("TASTE DESCRIPTIONS")
for taste, desc in taste_descriptions.items():
    word_count = len(desc.split())
    token_count = count_tokens(desc)
    print(f"[{taste}] {word_count} words / {token_count} tokens\n")

print("EMOTION DESCRIPTIONS")
for emotion, desc in emotion_descriptions.items():
    word_count = len(desc.split())
    token_count = count_tokens(desc)
    print(f"[{emotion}] {word_count} words / {token_count} tokens\n")

TASTE DESCRIPTIONS
[0] 40 words / 71 tokens

[1] 30 words / 48 tokens

[2] 40 words / 68 tokens

[3] 48 words / 76 tokens

[4] 27 words / 46 tokens

EMOTION DESCRIPTIONS
[0] 28 words / 41 tokens

[1] 29 words / 45 tokens

[2] 33 words / 52 tokens

[3] 27 words / 43 tokens

[4] 27 words / 39 tokens

[5] 36 words / 53 tokens

[6] 31 words / 50 tokens

[7] 26 words / 38 tokens

[8] 26 words / 47 tokens

[9] 28 words / 44 tokens

[10] 28 words / 42 tokens

[11] 27 words / 39 tokens



In [ ]:
import pickle

# Create separate dictionaries > code gotten from asking claude for saving the descriptions as dictionary
taste_descriptions = {}
for idx, row in taste_desc.iterrows():
    name = row.get('Unnamed: 0', f"taste_{idx}")
    taste_descriptions[name] = row['description']

emotion_descriptions = {}
for idx, row in emotion_desc.iterrows():
    name = row.get('Unnamed: 0', f"emotion_{idx}")
    emotion_descriptions[name] = row['description']

# Save separately
with open('/content/drive/MyDrive/dataset/taste_descriptions.pkl', 'wb') as f:
    pickle.dump(taste_descriptions, f)

with open('/content/drive/MyDrive/dataset/emotion_descriptions.pkl', 'wb') as f:
    pickle.dump(emotion_descriptions, f)

print(f"✓ Saved {len(taste_descriptions)} taste descriptions")
print(f"✓ Saved {len(emotion_descriptions)} emotion descriptions")

✓ Saved 5 taste descriptions
✓ Saved 12 emotion descriptions


Taste Descriptions:

1. A book cover that features Low (Curved, Smooth) angularity, High (Spherical, Circular) roundness, High (Reflectional/Rotational) symmetry, Low to Medium elements, Pink / Red hues, High (Light/Pastel) brightness, High (Vivid/Neon) - Candy like saturation, Smooth / Soft textures, Low (Simple/Clean) complexity.

2. A book cover that features High (Sharp, Spiky, Triangular) angularity, Low (Asymmetrical) symmetry, Many (High Clutter) elements, Green / Yellow hues, High (Vivid/Neon) saturation, Rough / Gritty / Jagged textures.

3. A book cover that features Low to Medium angularity, Low (Flat, Linear) roundness, Low to Medium symmetry, Few (Minimalist) - if simple/clean elements, Blue / White hues, High (Light/Pastel) brightness, Low (Desaturated/Grey) saturation, Crystalline / Rough textures, Low (Simple/Clean) complexity.

4. A book cover that features Very High (Sharp, Spiky, Triangular) angularity, Low roundness, Very Low (Asymmetrical) symmetry, Many (High Clutter) elements, Green / Yellow (if olive/ dark green)

Black / Brown / Purple hues, Low (Dark/Shadowy) brightness, Low to Medium saturation, Rough / Uneven textures, High (Intricate/Busy) complexity.

5. A book cover that features Low (Curved, Smooth) angularity, Black / Brown / Purple (if brown/ umami red) hues, Low (Dark/Shadowy) brightness, Medium saturation, High (Intricate/Busy) complexity.

Emotion Descriptions:

1. A book cover that features Low to Medium angularity, High roundness, Medium symmetry, Medium elements, Yellow, Orange hues, High brightness, High saturation, Smooth/ Soft textures, Medium (Playful) complexity.

2. A book cover that features Low angularity, Very High roundness, High (D8/Radial) symmetry, Low to Medium elements, Pink, Warm Red hues, High brightness, Medium saturation, Smooth/Creamy textures, Low complexity.

3. A book cover that features High (Spiky/Starbursts) angularity, Low roundness, Low or Radial symmetry, High elements, Neon Green, Bright Yellow, High Contrast hues, High brightness, High saturation, Sharp/Rough textures, High (Unexpected elements) complexity.

4. A book cover that features Low angularity, High roundness, High symmetry, Few (Minimalist) elements, Light Blue, White hues, High brightness, Low (Pastel) saturation, Smooth/Silky textures, Low complexity.

5. A book cover that features Low angularity, Medium roundness, High symmetry, Low elements, Beige, Warm Earth hues, Medium-High brightness, Medium saturation, Matte / Natural textures, Low complexity.

6. A book cover that features Very Low (Flat / Horizontal) angularity, Medium roundness, Medium symmetry, Very Low (Spares) elements, Beige, Pale Grey, Cream hues, Medium brightness, Very Low (Desaturated) saturation, Matte / Smooth textures, Minimal complexity.

7. A book cover that features High (Spiky) angularity, Low roundness, Low (Asymmetrical) symmetry, High (Cluttered) elements, Black, Purple, Grey hues, Very Low brightness, Low or High saturation, Rough/Gritty textures, High complexity.

8. A book cover that features High angularity, Low roundness, Low symmetry, High elements, Red, Black hues, Low to Medium brightness, High saturation, Rough/Scratchy textures, High complexity.

9. A book cover that features High angularity, Low roundness, Low (Unbalanced) symmetry, High elements, Clashing (e.g., Orange/Green) hues, Medium brightness, High saturation, Gritty/Noise textures, High complexity.

10. A book cover that features Low (Droopy) angularity, Medium roundness, Medium symmetry, Low (Empty) elements, Grey, Desaturated Blue hues, Low brightness, Very Low saturation, Flat/Matte textures, Low complexity.

11. A book cover that features Medium angularity, Medium roundness, Low (Asymmetrical) symmetry, Medium elements, Brown, Olive Green hues, Low brightness, Low to Medium saturation, Uneven/Sticky textures, Medium complexity.

12. A book cover that features Low angularity, Low roundness, High (Repetitive) symmetry, Low elements, Grey, Beige hues, Medium brightness, Very Low saturation, Flat textures, Very Low complexity.
